In [ ]:
mol_type = "6r7m"
n_mol = 3
T = 3.0
x = "Vector{Float64}([-162.86403396472377,66.25812707219484,85.96428200725117,37.128702469074035,99.23048857016123,77.50052452831233,-66.51419125876681,58.26457156376605,-4.441888460397806,29.082309905491627,118.81254532871007,74.20493968337979,40.67136368911156,16.916521934017293,61.411849022855314,47.90894046438104,136.7255328790018,73.12205747719372])"
comment = "3_86_extension"
# x = "Vector{Float64}([])"
# comment = ""
# T = 1.618
rs = 1.4
η = 0.3665
σ_r = 0.4
σ_t = 1.25
overlap_jump = 0.0
overlap_slope = 1.1
delaunay_eps = 100.0
bnds = 150.0

simulation_time_minutes = 18.0 * 60.0

720.0

In [2]:
input_specifier = "rwm_ma_$(n_mol)_$(mol_type)"

simulations_per_combination = 200

open("../configs/$(input_specifier)_config.txt", "w") do io
    i = 1
    println(io,"ArrayTaskID input_string")
    output_directory = "simulation_output/$(input_specifier)/"
    for _ in 0:simulations_per_combination-1
        name = "$(i)_$(input_specifier)"
        input_string = escape_string("name=\"$name\";x=$(x);T=$(T);rs=$rs;η=$η;σ_t=$σ_t;σ_r=$σ_r;overlap_jump=$overlap_jump;overlap_slope=$overlap_slope;bnds=$bnds;n_mol=$n_mol;mol_type=\"$mol_type\";simulation_time_minutes=$simulation_time_minutes;output_directory=\"$output_directory\";delaunay_eps=$delaunay_eps;comment=\"$comment\"")
        println(io, "$i $input_string")
        i += 1
    end
end

In [3]:
total_simulations = length(readlines("../configs/$(input_specifier)_config.txt")) - 1

100

In [4]:
hours = Int(round(simulation_time_minutes / 60.0))
days = hours ÷ 24
remaining_hours = hours % 24
remaining_hours_string = remaining_hours < 10 ? "0$(remaining_hours)" : string(remaining_hours)
buffer_time_string = simulation_time_minutes < 5 ? "0$(Int(simulation_time_minutes)+2)" : "30"

open("../$(input_specifier)_script.job", "w") do io
    println(io, "#!/bin/bash")
    println(io, "#SBATCH --job-name=SolvationSimulations")
    println(io, "#SBATCH --time=0$(days)-$(remaining_hours_string):$(buffer_time_string)")
    println(io, "#SBATCH --ntasks=1")
    println(io, "#SBATCH --cpus-per-task=1")
    println(io, "#SBATCH --array=1-$(total_simulations)")
    println(io, "#SBATCH --chdir=/work/spirandelli/MorphoMolHPC/")
    println(io, "#SBATCH -o ./job_log/$(input_specifier)/%a.out # STDOUT")
    println(io, "")
    println(io, "export http_proxy=http://proxy2.uni-potsdam.de:3128 #Setting proxy, due to lack of Internet on compute nodes.")
    println(io, "export https_proxy=http://proxy2.uni-potsdam.de:3128")
    println(io, "")
    println(io, "module purge")
    println(io, "module load lang/Julia/1.7.3-linux-x86_64")
    println(io, "")
    println(io, "# Specify the path to the config file")
    println(io, "config=hpc_scripts/configs/$(input_specifier)_config.txt")
    println(io, "")
    println(io, "# Extract the variables from config file")
    println(io, "config_string=\$(awk -v ArrayTaskID=\$SLURM_ARRAY_TASK_ID '\$1==ArrayTaskID {print \$2}' \$config)")
    println(io, "")
    println(io, "julia -e \"$(escape_string("include(\"julia_scripts/rwm_ma_call.jl\"); rwm_ma_call(\"\$config_string\")"))\"")
    println(io, "")
    println(io, "# copy back results")
    println(io, "mkdir -p ~/output/ && cp -r simulation_output/* ~/output/")
end